# 17. Train Exposure Ratio 실험

red scratch sample의 학습 노출 비율을 바꾸고, 동일 조합과 다른 조합의 추론 성능 변화를 비교합니다.

이 노트북의 결론은 `exposure_ratio_metrics_b0.csv`의 실제 추론 결과에서 계산됩니다.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "ch2_utils.py").exists():
    matches = (
        list(Path.cwd().glob("Deeplearning/*/2-1장/ch2_utils.py"))
        + list(Path.cwd().glob("Deeplearning/*/2장/ch2_utils.py"))
        + list(Path.cwd().glob("**/ch2_utils.py"))
    )
    NOTEBOOK_DIR = matches[0].parent if matches else Path("Deeplearning") / "Vision 응용" / "2-1장"
sys.path.append(str(NOTEBOOK_DIR))

from ch2_utils import *

paths = find_paths()
set_korean_font()
set_seed(7)
samples = load_samples(paths.data_root)
paths

## 17-1. Exposure ratio manifest 확인

In [ ]:
exposure_summary = create_exposure_manifests(
    samples,
    ratios=[0.0, 0.1, 0.25, 0.5, 0.75],
    target_color="red",
    target_defect="scratch",
    train_size=360,
    eval_per_combo=12,
    seed=7,
    runs_root=paths.runs_root,
)
display(exposure_summary)

## 17-2. Ratio별 SegFormer 학습과 추론

In [ ]:
EPOCHS = 2
BATCH_SIZE = 8
LR = 1e-3

all_sample_metrics = []
for _, row in exposure_summary.iterrows():
    ratio = float(row["exposure_ratio"])
    run_dir = paths.runs_root / "exposure_ratio_b0" / row["run_name"]
    if not (run_dir / "sample_metrics.csv").exists():
        print("training:", row["run_name"])
        train_segformer_experiment(
            train_manifest=row["train_manifest"],
            eval_manifest=row["eval_manifest"],
            run_dir=run_dir,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            lr=LR,
            seed=7,
            augment=False,
            model_name=SEGFORMER_B0_MODEL_NAME,
            use_pretrained=True,
        )
    sample_m, group_m, class_m = load_run_metrics(run_dir)
    sample_m["exposure_ratio"] = ratio
    sample_m["run_name"] = row["run_name"]
    all_sample_metrics.append(sample_m)

exposure_metrics = pd.concat(all_sample_metrics, ignore_index=True)
out_path = paths.runs_root / "exposure_ratio_metrics_b0.csv"
exposure_metrics.to_csv(out_path, index=False, encoding="utf-8-sig")
display(exposure_metrics.head())
print(out_path)

## 17-3. 노출 비율별 성능 곡선

In [ ]:
summary = (
    exposure_metrics.groupby(["exposure_ratio", "eval_combo"])["target_dice"]
    .mean()
    .reset_index()
)
fig, ax = plt.subplots(figsize=(9, 5))
for combo, part in summary.groupby("eval_combo"):
    part = part.sort_values("exposure_ratio")
    ax.plot(part["exposure_ratio"], part["target_dice"], marker="o", label=combo)
ax.set_xlabel("red scratch train exposure ratio")
ax.set_ylabel("target Dice")
ax.set_ylim(0, 1)
ax.grid(alpha=0.25)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
fig.savefig(paths.runs_root / "17_exposure_ratio_curve_b0.png", dpi=150)
plt.show()
display(summary)

## 17-4. 노출 비율 실험 결론

In [ ]:
conclusions = []
for combo, part in summary.groupby("eval_combo"):
    part = part.sort_values("exposure_ratio")
    first = part.iloc[0]["target_dice"]
    last = part.iloc[-1]["target_dice"]
    delta = last - first
    conclusions.append({"eval_combo": combo, "dice_at_min_ratio": first, "dice_at_max_ratio": last, "delta": delta})
conclusions = pd.DataFrame(conclusions).sort_values("delta", ascending=False)
conclusions.to_csv(paths.runs_root / "17_exposure_ratio_conclusions_b0.csv", index=False, encoding="utf-8-sig")
display(conclusions)
best = conclusions.iloc[0]
worst = conclusions.iloc[-1]
print(
    f"결론: red scratch 노출 비율 증가에 가장 크게 반응한 조합은 {best['eval_combo']} (delta={best['delta']:.3f})이고, "
    f"가장 개선이 작거나 악화된 조합은 {worst['eval_combo']} (delta={worst['delta']:.3f})입니다."
)